In [1]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path().resolve().parents[0]))

In [2]:
from base_data_pipeline import get_processed_data
from base_data_pipeline import load_dataset

In [3]:
train_texts, test_texts, train_labels, test_labels = get_processed_data()

In [4]:
df = load_dataset()

In [5]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df.head()


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
class SMSDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = SMSDataset(train_encodings, train_labels)
test_dataset = SMSDataset(test_encodings, test_labels)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(torch.cuda.is_available)
print(device)

<function is_available at 0x79ada1d354e0>
cuda


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import copy

num_epochs = 10
patience = 2

best_f1 = 0
epochs_no_improve = 0
best_model_state = None

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):

        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)

        loss = outputs.loss
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Training Loss: {avg_loss}")

    # evaluation
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in test_loader:

            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(batch["labels"].cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average="binary"
    )

    print("Accuracy:", accuracy)
    print("F1 Score:", f1)

    if f1 > best_f1:

        best_f1 = f1
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0

        print("Improvement detected. Saving model.")

    else:

        epochs_no_improve += 1
        print("No improvement. Patience counter:", epochs_no_improve)

    if epochs_no_improve >= patience:

        print("\nEarly stopping triggered.")
        break


model.load_state_dict(best_model_state)

print("\nBest model restored with F1:", best_f1)

100%|██████████| 279/279 [00:46<00:00,  6.01it/s]



Epoch 1 Training Loss: 0.06902589283943657
Accuracy: 0.9910313901345291
F1 Score: 0.9659863945578231
Improvement detected. Saving model.


100%|██████████| 279/279 [00:47<00:00,  5.82it/s]



Epoch 2 Training Loss: 0.020649275293813243
Accuracy: 0.989237668161435
F1 Score: 0.9586206896551724
No improvement. Patience counter: 1


100%|██████████| 279/279 [00:49<00:00,  5.58it/s]



Epoch 3 Training Loss: 0.006782022762879242
Accuracy: 0.9910313901345291
F1 Score: 0.9662162162162162
Improvement detected. Saving model.


100%|██████████| 279/279 [00:49<00:00,  5.63it/s]



Epoch 4 Training Loss: 0.009332415379663532
Accuracy: 0.9910313901345291
F1 Score: 0.9657534246575342
No improvement. Patience counter: 1


100%|██████████| 279/279 [00:49<00:00,  5.64it/s]



Epoch 5 Training Loss: 0.0050486747747387085
Accuracy: 0.9901345291479821
F1 Score: 0.9627118644067797
No improvement. Patience counter: 2

Early stopping triggered.

Best model restored with F1: 0.9662162162162162


In [ ]:
!cp /content/drive/MyDrive/Colab\ Notebooks/evaluation_utils.py /content

In [ ]:
import evaluation_utils

In [ ]:
results = evaluation_utils.evaluate_model(true_labels, predictions)

In [ ]:
results

{'Accuracy': 0.9901345291479821,
 'Precision': 0.9726027397260274,
 'Recall': 0.9530201342281879,
 'F1 Score': 0.9627118644067797,
 'Confusion Matrix': array([[962,   4],
        [  7, 142]])}